# ATIIS p0 sweep: comparative analysis

Manifest-driven analysis of the belief (`p0`) sweep and its OFAT validation arms.

- **Headline** (the `baseline` arm, all models x scenarios): does the LLM blame
  curve track the formal Shapley oracle? Shape (rank) agreement, an explicit
  non-monotonicity test, the blame-vs-p0 curves, and a variance decomposition
  (parameter vs wording).
- **Validation** (s01 only, per arm): does the curve shift the way the oracle
  shifts when one variable changes? `vote` (Wilcoxon + McNemar), `alpha`
  (peak-location shift + Mann-Whitney), `cost`/`stakes` (Mann-Whitney + slope vs
  the model's own inferred cost). Plus `oracle_at_inferred` agreement everywhere.

All tables are written to `analysis/tables/` (CSV + LaTeX) and figures to
`analysis/figures/` for embedding in the report.

## 0  Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Make `engine` and `analysis` importable whether the notebook runs from the
# experiment root or from analysis/.
HERE = Path.cwd()
EXP_ROOT = HERE if (HERE / "engine").exists() else HERE.parent
sys.path.insert(0, str(EXP_ROOT))

from analysis.loader import load_runs, P0_LEVEL_ORDER

FIG = EXP_ROOT / "analysis" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
TAB = EXP_ROOT / "analysis" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(0)

In [ ]:
def spearman(x, y):
    return stats.spearmanr(x, y).statistic

def bootstrap_ci(stat_fn, *arrays, n=2000, alpha=0.05):
    """Percentile bootstrap CI for a statistic over paired arrays."""
    arrays = [np.asarray(a, float) for a in arrays]
    m = len(arrays[0]); idx = np.arange(m); vals = []
    for _ in range(n):
        s = RNG.choice(idx, m, replace=True)
        try:
            vals.append(stat_fn(*[a[s] for a in arrays]))
        except Exception:
            pass
    if not vals:
        return np.nan, np.nan
    return tuple(np.nanpercentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)]))

def bh(pvals):
    """Benjamini-Hochberg adjusted p-values (scipy >= 1.11, with a fallback)."""
    p = np.asarray(pvals, float)
    try:
        return stats.false_discovery_control(p, method="bh")
    except Exception:
        order = np.argsort(p); ranked = np.empty_like(p); m = len(p); cummin = 1.0
        for i in range(m - 1, -1, -1):
            cummin = min(cummin, p[order[i]] * m / (i + 1))
            ranked[order[i]] = cummin
        return ranked

def save_table(df, name):
    df.to_csv(TAB / f"{name}.csv", index=False)
    fmt = lambda v: f"{v:.3f}" if isinstance(v, float) else str(v)
    (TAB / f"{name}.tex").write_text(df.to_latex(index=False, float_format="%.3f"))
    print(f"wrote tables/{name}.csv + .tex")
    return df

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")
    print(f"wrote figures/{name}.png")
    plt.show()

## 1  Load runs

In [ ]:
df = load_runs(sweep="sweep1_p0")
df["arm"] = df["arm"].astype(str)
base = df[df["arm"] == "baseline"].copy()                  # headline
print("calls per (model, scenario, arm):")
print(df.groupby(["model_slug", "scenario", "arm"]).size())

## 2  Headline: does LLM blame track the oracle?

### 2.1  Rank (shape) agreement with the oracle

In [ ]:
rows = []
for (m, s), g in base.groupby(["model_slug", "scenario"]):
    blame = g["blameworthiness"].to_numpy(float)
    o_ref = g["oracle_at_reference"].to_numpy(float)
    o_inf = g["oracle_at_inferred"].to_numpy(float)
    rho = spearman(blame, o_ref)
    lo, hi = bootstrap_ci(spearman, blame, o_ref)
    rows.append(dict(
        model=m, scenario=s, n=len(g),
        spearman_ref=rho, ci_lo=lo, ci_hi=hi,
        p=stats.spearmanr(blame, o_ref).pvalue,
        spearman_inferred=spearman(blame, o_inf),
    ))
head = pd.DataFrame(rows)
head["p_bh"] = bh(head["p"])
save_table(head, "headline_spearman")
head

### 2.2  Non-monotonicity test

The theory predicts a non-monotonic peak in blame vs `p0`. A bare Spearman would
miss that. Fit `blame ~ rank + rank^2` and test the quadratic term; compare the
fitted LLM peak rank to the oracle's peak rank.

In [ ]:
def quad_test(g):
    x = g["level_rank"].to_numpy(float); y = g["blameworthiness"].to_numpy(float)
    X = np.column_stack([np.ones_like(x), x, x ** 2])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta; dof = len(y) - 3
    sigma2 = (resid @ resid) / dof
    se = np.sqrt(sigma2 * np.diag(np.linalg.inv(X.T @ X)))
    t = beta[2] / se[2]
    pval = 2 * stats.t.sf(abs(t), dof)
    peak = -beta[1] / (2 * beta[2]) if beta[2] != 0 else np.nan
    return beta[2], pval, peak

rows = []
for (m, s), g in base.groupby(["model_slug", "scenario"]):
    b2, pv, peak = quad_test(g)
    oracle_peak = g.groupby("level_rank")["oracle_at_reference"].mean().idxmax()
    rows.append(dict(model=m, scenario=s, quad_coef=b2, quad_p=pv,
                     llm_peak_rank=peak, oracle_peak_rank=oracle_peak))
nm = pd.DataFrame(rows)
nm["quad_p_bh"] = bh(nm["quad_p"])
save_table(nm, "headline_nonmonotonicity")
nm

### 2.3  Blame-vs-p0 curves with oracle overlay

In [ ]:
levels = P0_LEVEL_ORDER
for scenario, gs in base.groupby("scenario"):
    models = sorted(gs["model_slug"].unique())
    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4),
                             sharey=True, squeeze=False)
    for ax, m in zip(axes[0], models):
        g = gs[gs["model_slug"] == m]
        gm = g.groupby("level_rank")["blameworthiness"].agg(["mean", "sem"])
        ax.errorbar(gm.index, gm["mean"], yerr=gm["sem"], marker="o", color="tab:blue",
                    label="LLM blame")
        ax.set_title(m); ax.set_xticks(range(len(levels)))
        ax.set_xticklabels(levels, rotation=45, ha="right")
        ax2 = ax.twinx()
        oref = g.groupby("level_rank")["oracle_at_reference"].mean()
        oinf = g.groupby("level_rank")["oracle_at_inferred"].mean()
        ax2.plot(oref.index, oref.values, "--", color="k", label="oracle (ref)")
        ax2.plot(oinf.index, oinf.values, ":", color="gray", label="oracle (inferred)")
        ax2.set_ylabel("oracle Shapley blame")
    axes[0][0].set_ylabel("LLM blameworthiness (0-100)")
    fig.suptitle(f"Headline blame vs p0  |  scenario={scenario}")
    savefig(f"headline_curves_{scenario}")

### 2.4  Variance decomposition (parameter vs wording)

Dependency-light ICC-style summary: the share of total blame variance explained
by the parameter (`level`) versus the phrasing within a level. A small phrasing
share means the signal is from the parameter, not the wording.

In [ ]:
def variance_shares(g):
    y = g["blameworthiness"].to_numpy(float)
    gm = y.mean(); sst = ((y - gm) ** 2).sum()
    if sst == 0:
        return np.nan, np.nan
    ss_level = sum(len(gg) * (gg["blameworthiness"].mean() - gm) ** 2
                   for _, gg in g.groupby("level_rank"))
    ss_phr = 0.0
    for _, gl in g.groupby("level_rank"):
        lm = gl["blameworthiness"].mean()
        for _, gp in gl.groupby("phrasing_idx"):
            ss_phr += len(gp) * (gp["blameworthiness"].mean() - lm) ** 2
    return ss_level / sst, ss_phr / sst

rows = []
for (m, s), g in base.groupby(["model_slug", "scenario"]):
    el, ep = variance_shares(g)
    rows.append(dict(model=m, scenario=s, var_param=el, var_phrasing=ep,
                     var_resid=1 - el - ep))
vd = pd.DataFrame(rows)
save_table(vd, "headline_variance_shares")
vd

In [ ]:
# Optional: a formal mixed model if statsmodels is available (not required).
try:
    import statsmodels.formula.api as smf
    g = base[base["scenario"] == base["scenario"].iloc[0]].copy()
    md_fit = smf.mixedlm("blameworthiness ~ C(level)", g, groups=g["phrasing_idx"]).fit()
    print(md_fit.summary())
except Exception as exc:
    print("statsmodels MixedLM skipped (use section 2.4 instead):", exc)

## 3  Validation arms (s01): does the curve shift like the framework?

Each arm changes one held variable. We check the LLM curve moves the way the
oracle curve moves.

In [ ]:
S = "s01_health"
val = df[df["scenario"] == S].copy()
ARMS_PRESENT = sorted(val["arm"].unique())
print("arms present in", S, ":", ARMS_PRESENT)

### 3.1  Own vote (`vote_yes`)

H0: the yes-voter gets zero blame. Wilcoxon signed-rank pairs no vs yes by
(level, phrasing, rep); McNemar tests the binary `blame==0` for the yes-voter;
the blame ratio is compared to the oracle's (about 3.3).

In [ ]:
rows = []
for m, g in val.groupby("model_slug"):
    b = g[g["arm"] == "baseline"]; y = g[g["arm"] == "vote_yes"]
    if y.empty:
        continue
    key = ["level", "phrasing_idx", "rep"]
    merged = b.merge(y, on=key, suffixes=("_no", "_yes"))
    if merged.empty:
        continue
    w = stats.wilcoxon(merged["blameworthiness_no"], merged["blameworthiness_yes"],
                       zero_method="zsplit")
    no0 = merged["blameworthiness_no"] == 0; yes0 = merged["blameworthiness_yes"] == 0
    bb = int((~no0 & yes0).sum()); cc = int((no0 & ~yes0).sum())
    chi = (abs(bb - cc) - 1) ** 2 / (bb + cc) if (bb + cc) else np.nan
    mcp = stats.chi2.sf(chi, 1) if (bb + cc) else np.nan
    ratio = merged["blameworthiness_no"].mean() / max(merged["blameworthiness_yes"].mean(), 1e-9)
    rows.append(dict(model=m, n_pairs=len(merged),
                     median_no=merged["blameworthiness_no"].median(),
                     median_yes=merged["blameworthiness_yes"].median(),
                     wilcoxon_p=w.pvalue, mcnemar_b=bb, mcnemar_c=cc, mcnemar_p=mcp,
                     blame_ratio=ratio, oracle_ratio=3.3))
vote = pd.DataFrame(rows)
if not vote.empty:
    vote["wilcoxon_p_bh"] = bh(vote["wilcoxon_p"])
    save_table(vote, "validation_vote")
vote

### 3.2  Pressure effectiveness (`alpha_negligible`, `alpha_strong`)

The framework moves the peak location of the p0 curve as alpha changes. Compare
the LLM peak shift to the oracle peak shift; Mann-Whitney tests the overall
distribution shift vs baseline.

In [ ]:
rows = []
for m, g in val.groupby("model_slug"):
    base_g = g[g["arm"] == "baseline"]
    for arm in ["alpha_negligible", "alpha_strong"]:
        a = g[g["arm"] == arm]
        if a.empty or base_g.empty:
            continue
        u = stats.mannwhitneyu(base_g["blameworthiness"], a["blameworthiness"])
        rows.append(dict(
            model=m, arm=arm,
            llm_peak_base=base_g.groupby("level_rank")["blameworthiness"].mean().idxmax(),
            llm_peak_arm=a.groupby("level_rank")["blameworthiness"].mean().idxmax(),
            oracle_peak_base=base_g.groupby("level_rank")["oracle_at_reference"].mean().idxmax(),
            oracle_peak_arm=a.groupby("level_rank")["oracle_at_reference"].mean().idxmax(),
            mwu_p=u.pvalue,
            spearman_inferred=spearman(a["blameworthiness"], a["oracle_at_inferred"]),
        ))
alpha = pd.DataFrame(rows)
if not alpha.empty:
    alpha["mwu_p_bh"] = bh(alpha["mwu_p"])
    save_table(alpha, "validation_alpha")
alpha

### 3.3  Cost and stakes (`cost_trivial`, `stakes_high`)

Mann-Whitney vs baseline, plus the slope of blame against the model's own
inferred cost (the `c_sw/N` reading), which is the assumption-free cost-discount
check.

In [ ]:
rows = []
for m, g in val.groupby("model_slug"):
    base_g = g[g["arm"] == "baseline"]
    for arm in ["cost_trivial", "stakes_high"]:
        a = g[g["arm"] == arm]
        if a.empty or base_g.empty:
            continue
        u = stats.mannwhitneyu(base_g["blameworthiness"], a["blameworthiness"])
        sub = a.dropna(subset=["inferred_cost"])
        slope = pslope = np.nan
        if len(sub) > 2 and sub["inferred_cost"].nunique() > 1:
            lr = stats.linregress(sub["inferred_cost"], sub["blameworthiness"])
            slope, pslope = lr.slope, lr.pvalue
        rows.append(dict(model=m, arm=arm,
                         median_base=base_g["blameworthiness"].median(),
                         median_arm=a["blameworthiness"].median(), mwu_p=u.pvalue,
                         blame_vs_inferredcost_slope=slope, slope_p=pslope,
                         spearman_inferred=spearman(a["blameworthiness"], a["oracle_at_inferred"])))
cost = pd.DataFrame(rows)
if not cost.empty:
    cost["mwu_p_bh"] = bh(cost["mwu_p"])
    save_table(cost, "validation_cost_stakes")
cost

### 3.4  Validation curves: baseline vs each arm

In [ ]:
arm_order = [a for a in ["baseline", "vote_yes", "alpha_negligible", "alpha_strong",
                         "cost_trivial", "stakes_high"] if a in ARMS_PRESENT]
models = sorted(val["model_slug"].unique())
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4),
                         sharey=True, squeeze=False)
for ax, m in zip(axes[0], models):
    g = val[val["model_slug"] == m]
    for arm in arm_order:
        a = g[g["arm"] == arm]
        if a.empty:
            continue
        gm = a.groupby("level_rank")["blameworthiness"].mean()
        ax.plot(gm.index, gm.values, marker="o", label=arm)
    ax.set_title(m); ax.set_xticks(range(len(P0_LEVEL_ORDER)))
    ax.set_xticklabels(P0_LEVEL_ORDER, rotation=45, ha="right")
axes[0][0].set_ylabel("LLM blameworthiness (0-100)")
axes[0][-1].legend(fontsize=8, loc="upper right")
fig.suptitle(f"Validation: blame vs p0 by arm  |  scenario={S}")
savefig("validation_curves")

## 4  Outputs

Tables in `analysis/tables/` (CSV + LaTeX), figures in `analysis/figures/`.
Pull these directly into the report.

In [ ]:
print("tables:", sorted(p.name for p in TAB.glob("*.csv")))
print("figures:", sorted(p.name for p in FIG.glob("*.png")))